YOLOv7 Object Detection

**Tasks**
1. Detect players in football video
2. Detect sports ball in football video
3. Real-time webcam detection with a mis-detection screenshot

Runtime: **GPU** (Runtime → Change runtime type → T4 or A100).

## 0. Verify GPU

In [ ]:
!nvidia-smi

## 1. Clone the YOLOv7 object-tracking repo

In [ ]:
%cd /content
!git clone https://github.com/RizwanMunawar/yolov7-object-tracking.git
%cd yolov7-object-tracking

## 2. Install dependencies
Colab already has PyTorch + CUDA. Install the rest from the repo's requirements file.

In [ ]:
!pip install -q -r requirements.txt
# SORT tracker uses np.int which is removed in numpy>=1.24. Patch it:
!sed -i 's/np\.int)/int)/g' sort.py || true
!pip install -q 'numpy<2' 'opencv-python-headless' 'filterpy' 'scikit-image' 'lap'

## 3. Download pre-trained YOLOv7 weights

In [ ]:
!wget -q https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt -O yolov7.pt
!ls -lh yolov7.pt

## 4. Upload your `football.mp4`
Run the cell and pick `football.mp4` from your computer. (Alternative: mount Google Drive with `from google.colab import drive; drive.mount('/content/drive')` and copy it over.)

In [ ]:
from google.colab import files
uploaded = files.upload()  # select football.mp4

## 5. Trim to a ~45-second clip with action
The full video is ~5 minutes. Trim it to a chunk with visible players and ball movement. Adjust `-ss` (start time) and `-t` (duration) as you like; assignment requires ≥ 30 seconds.

In [ ]:
!ffmpeg -y -ss 00:01:00 -i football.mp4 -t 45 -c:v libx264 -an football_clip.mp4
!ls -lh football_clip.mp4

## Task 1 — Detect players only
COCO class `0` = `person`. Using `detect_and_track.py` adds bounding-box IDs, matching the sample output in the PDF.

In [ ]:
!python detect_and_track.py \
  --weights yolov7.pt \
  --source football_clip.mp4 \
  --classes 0 \
  --device 0 \
  --name task1_players \
  --no-trace 2>&1 | tee task1_log.txt

## Task 2 — Detect the sports ball
COCO class `32` = `sports ball`. The ball is small and fast, so lower the confidence threshold a bit.

In [ ]:
!python detect.py \
  --weights yolov7.pt \
  --source football_clip.mp4 \
  --classes 32 \
  --conf-thres 0.15 \
  --device 0 \
  --name task2_ball \
  --no-trace 2>&1 | tee task2_log.txt

### Re-encode the output videos so they play inline
YOLOv7 writes MP4 with a codec that Colab's preview sometimes can't play. Re-encode to H.264 + faststart.

In [ ]:
!ffmpeg -y -i runs/detect/task1_players/football_clip.mp4 -vcodec libx264 -movflags +faststart task1_players.mp4
!ffmpeg -y -i runs/detect/task2_ball/football_clip.mp4   -vcodec libx264 -movflags +faststart task2_ball.mp4
!ls -lh task1_players.mp4 task2_ball.mp4

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_video(path, width=640):
    data = b64encode(open(path, 'rb').read()).decode()
    return HTML(f'<video width={width} controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

show_video('task1_players.mp4')

In [ ]:
show_video('task2_ball.mp4')

## Task 3 — Webcam detection with a visible mis-detection
Colab's VM has no webcam, so we use a browser JavaScript snippet to grab a frame from your laptop camera, then run YOLOv7 on it.

**Idea for mis-detections**: point the webcam at a stuffed animal, a photo on your phone, a coat on a chair, or a mug with a logo — YOLO often mislabels these.

You can re-run this cell multiple times until you capture a frame where the model clearly gets it wrong.

In [ ]:
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='webcam.jpg', quality=0.9):
    js = Javascript('''
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const capture = document.createElement('button');
          capture.textContent = 'Capture';
          div.appendChild(capture);
          const video = document.createElement('video');
          video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => capture.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js(f'takePhoto({quality})')
    with open(filename, 'wb') as f:
        f.write(b64decode(data.split(',')[1]))
    return filename

img_path = take_photo('webcam.jpg')
print('Saved', img_path)

In [ ]:
!python detect.py \
  --weights yolov7.pt \
  --source webcam.jpg \
  --device 0 \
  --name task3_webcam \
  --no-trace 2>&1 | tee task3_log.txt

from IPython.display import Image
Image('runs/detect/task3_webcam/webcam.jpg')

## 6. Collect everything for submission

In [ ]:
!mkdir -p submission
!cp task1_players.mp4   submission/task1_players.mp4
!cp task2_ball.mp4      submission/task2_ball.mp4
!cp runs/detect/task3_webcam/webcam.jpg submission/task3_webcam.jpg
# Combine all the terminal logs into one txt file
!cat task1_log.txt task2_log.txt task3_log.txt > submission/commands_and_output.txt
!ls -lh submission/

In [ ]:
from google.colab import files
!zip -r submission.zip submission
files.download('submission.zip')

## Notes for your README
Cover these four things (from the assignment PDF):

1. **Command explanation.** `--weights yolov7.pt` loads the pre-trained model; `--source` is the input (video, image, or `0` for webcam); `--classes 0` filters detections to *person*, `--classes 32` filters to *sports ball* (COCO IDs); `--conf-thres` is the minimum confidence; `--device 0` uses the first GPU; `--no-trace` skips TorchScript tracing (avoids a PyTorch 2 warning); `--name` sets the output subfolder under `runs/detect/`.
2. **Problems observed.** e.g. the ball is often missed when occluded by players or motion-blurred; crowd members in the stands get detected as *person*; overlapping players merge into one box; IDs switch between frames when tracking.
3. **Reasons + improvements.** Small-object detection suffers at 640-px input (try `--img-size 1280`); the model was trained on COCO, not football broadcasts (fine-tune on the Kaggle football-analysis dataset); motion blur could be reduced with frame deblurring; use `yolov7-e6e.pt` or `yolov7-w6.pt` for higher accuracy; use tiled inference (SAHI) for tiny balls.
4. **Your understanding of YOLO.** Single-stage detector — one forward pass predicts class probabilities and bounding-box coordinates simultaneously. Divides the image into a grid, each grid cell predicts anchored boxes, NMS removes duplicates. Trained on COCO with 80 classes. YOLOv7 contributions: Extended-ELAN backbone, auxiliary head with coarse-to-fine supervision, model re-parameterization at inference time.